# RL Beat Generation — Phase 1 Discriminator Training

Trains `BeatDiscriminator` (`num_instruments=4, d_ff=128`) on 4-instrument slices of
Groove MIDI grids vs. synthetic negatives.  Produces `discriminator_phase1_v2.pt`,
which is uploaded to Drive and used by the PPO training notebook.

> **Runtime:** `Runtime → Change runtime type → T4 GPU`  
> **Run order:** Mount Drive first (Cell 2), then execute top-to-bottom.

## 1 · Setup

In [ ]:
!git clone -b atharv/ppo-discriminator https://github.com/Atharv-Girish-Chaudhary/rl-beat-generation.git
%cd rl-beat-generation
!pip install -e . -q

## 2 · Mount Drive + Copy Data

`groove_grids.npy` is too large to commit to git.  This cell pulls it from your
Drive so the training script can find it at `data/processed/groove_grids.npy`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
drive_path = "/content/drive/MyDrive/02_Coding_&_Projects/rl-beat-generation"

# Copy groove_grids.npy from Drive if not already present
os.makedirs("data/processed", exist_ok=True)
if not os.path.exists("data/processed/groove_grids.npy"):
    shutil.copy(f"{drive_path}/groove_grids.npy", "data/processed/groove_grids.npy")
    print("✅ Copied groove_grids.npy from Drive")
else:
    print("✅ groove_grids.npy already present")

import numpy as np
shape = np.load("data/processed/groove_grids.npy").shape
print(f"   Shape: {shape}  (expected (22965, 8, 16))")

## 3 · Device Check

In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    props = torch.cuda.get_device_properties(0)
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {props.total_memory / 1e9:.1f} GB")
else:
    device = "cpu"
    print("GPU  : none — training on CPU (will be slow)")

# System RAM via /proc/meminfo (always available on Colab)
with open('/proc/meminfo') as f:
    lines = {k.strip(): v.strip()
             for k, v in (l.split(':', 1) for l in f if ':' in l)}
total_kb = int(lines.get('MemTotal', '0 kB').split()[0])
avail_kb = int(lines.get('MemAvailable', '0 kB').split()[0])
print(f"RAM  : {total_kb/1e6:.1f} GB total, {avail_kb/1e6:.1f} GB available")
print(f"\nPyTorch: {torch.__version__}")

## 4 · Config

In [ ]:
EPOCHS     = 15
BATCH_SIZE = 64
LR         = 1e-3
SAVE_PATH  = "outputs/checkpoints/discriminator_phase1_v2.pt"

print(f"epochs={EPOCHS}, batch_size={BATCH_SIZE}, lr={LR}")
print(f"save_path={SAVE_PATH}")

## 5 · Train

Calls `train_discriminator()` inline so the config values above are forwarded.
Equivalent to running `!python scripts/train_discriminator.py` with those args.

Expected: validation accuracy should reach **≥ 0.75** within 15 epochs on a T4.

In [ ]:
import sys, matplotlib
sys.path.insert(0, '.')
matplotlib.use('Agg')   # headless during training; plots are saved to outputs/plots/

from scripts.train_discriminator import train_discriminator

train_discriminator(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
)

## 6 · Sanity Check

Reload the saved checkpoint and verify that a structured beat scores higher than silence.
This is the minimum bar the discriminator must clear before being used in PPO training.

In [ ]:
from beat_rl.models import BeatDiscriminator

L, T = 4, 16

disc = BeatDiscriminator(
    num_instruments=L, num_steps=T,
    d_model=64, num_heads=4, num_blocks=2, d_ff=128
).to(device)
disc.load_state_dict(torch.load(SAVE_PATH, map_location=device))
disc.eval()
print(f"Loaded {SAVE_PATH}")

# ── Pass 1: completely silent grid ────────────────────────────────────────────
silent = torch.zeros(1, L, T, device=device)
with torch.no_grad():
    logit_s, _ = disc(silent)
score_silent = torch.sigmoid(logit_s).item()

# ── Pass 2: structured beat (kick/snare/hihat/clap) ───────────────────────────
structured = torch.zeros(1, L, T)
structured[0, 0, [0, 4, 8, 12]] = 1.0          # kick   — quarter notes
structured[0, 1, [4, 12]]       = 1.0          # snare  — 2 & 4
structured[0, 2, list(range(0, 16, 2))] = 1.0  # hi-hat — 8th notes
structured[0, 3, [4, 12]]       = 1.0          # clap   — 2 & 4
with torch.no_grad():
    logit_r, _ = disc(structured.to(device))
score_structured = torch.sigmoid(logit_r).item()

print(f"\nSilent grid score    : {score_silent:.4f}  (expect < 0.5)")
print(f"Structured beat score: {score_structured:.4f}  (expect > silent)")

assert score_structured > score_silent, (
    f"Sanity check FAILED — structured ({score_structured:.4f}) "
    f"<= silent ({score_silent:.4f}). "
    "Re-train or check that val accuracy reached >= 0.75."
)
print("\n✅ Sanity check passed — discriminator is ready for PPO training.")

## 7 · Save to Drive

Copies the checkpoint back to Drive so it persists across Colab sessions
and is available to the PPO training notebook.

In [ ]:
import shutil, os

dest_dir = f"{drive_path}/checkpoints"
os.makedirs(dest_dir, exist_ok=True)

shutil.copy(
    "outputs/checkpoints/discriminator_phase1_v2.pt",
    f"{dest_dir}/discriminator_phase1_v2.pt",
)
print(f"✅ Saved to {dest_dir}/discriminator_phase1_v2.pt")